In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

print("drive exists:", Path("/content/drive").is_dir())
print("MyDrive exists:", Path("/content/drive/MyDrive").is_dir())

In [ ]:
# ============================================================
# ARC-v0.24 POST-PRIMARY REVIEWER AUDIT
#
# Purpose:
#   A1. Direct paired mechanism contrast:
#       representation H3abs - nprobe H3abs
#
#   A2. Feedback-family decomposition:
#       mean-only vs softmax-only H3abs
#
#   A3. MS MARCO signed-direction query-cluster bootstrap CI:
#       P(higher fidelity wins | H3abs > epsilon)
#
# IMPORTANT:
#   - NO retrieval rerun
#   - NO corpus encoding
#   - NO FAISS index build/load
#   - Reads ONLY completed endpoint parquet
#   - Post-primary robustness / reviewer audit
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ----------------------------
# Frozen constants
# ----------------------------
SEED = 20260824
BOOTSTRAP_REPS = 10_000
EPS_PRIMARY = 0.002

OLD_RUN_ID = "20260823-140056"

OUT = (
    Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-v0")
    / "msmarco-external-boundary-replication-v024"
    / OLD_RUN_ID
)

ENDPOINT_PATH = OUT / "v024_validation-full_endpoints.parquet"

assert OUT.is_dir(), f"Missing run dir:\n{OUT}"
assert ENDPOINT_PATH.is_file(), f"Missing endpoint file:\n{ENDPOINT_PATH}"

print("Loading:", ENDPOINT_PATH)

ep = pd.read_parquet(ENDPOINT_PATH)

required = {
    "query_id",
    "mechanism",
    "method",
    "H3_abs_slope",
    "H3_signed_slope",
    "final_signed_gap",
}
missing = required - set(ep.columns)
assert not missing, f"Missing columns: {missing}"

assert ep["query_id"].nunique() == 3490
assert set(ep["mechanism"].unique()) == {"representation", "nprobe"}

print("rows:", f"{len(ep):,}")
print("queries:", ep["query_id"].nunique())
print("mechanisms:", sorted(ep["mechanism"].unique()))
print("methods:", sorted(ep["method"].unique()))

# ============================================================
# Helpers
# ============================================================

def percentile_ci(x, reps=BOOTSTRAP_REPS, seed=0):
    """
    Simple query-level bootstrap for one value per query.
    """
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    n = len(x)
    assert n > 1

    rng = np.random.default_rng(seed)
    boots = np.empty(reps, dtype=np.float64)

    for b in range(reps):
        idx = rng.integers(0, n, size=n)
        boots[b] = x[idx].mean()

    lo, hi = np.quantile(boots, [0.025, 0.975])

    return {
        "mean": float(x.mean()),
        "ci95_low": float(lo),
        "ci95_high": float(hi),
        "n_queries": int(n),
    }


def cluster_bootstrap_event_fraction(
    df,
    event_mask_col,
    win_col,
    reps=BOOTSTRAP_REPS,
    seed=0,
):
    """
    Query-cluster bootstrap.

    Each sampled query carries ALL of its query-policy events.
    Point estimate is event-weighted conditional fraction, but
    uncertainty resamples at query level.

    This is appropriate because query-policy events are NOT treated
    as independent statistical observations.
    """

    work = df[
        ["query_id", event_mask_col, win_col]
    ].copy()

    # Aggregate counts within each query.
    q = (
        work.groupby("query_id", as_index=False)
        .agg(
            amp_events=(event_mask_col, "sum"),
            win_events=(win_col, "sum"),
        )
    )

    q = q[q["amp_events"] > 0].reset_index(drop=True)

    amp = q["amp_events"].to_numpy(np.float64)
    win = q["win_events"].to_numpy(np.float64)

    assert amp.sum() > 0

    point = float(win.sum() / amp.sum())

    rng = np.random.default_rng(seed)
    n = len(q)
    boots = np.empty(reps, dtype=np.float64)

    for b in range(reps):
        idx = rng.integers(0, n, size=n)
        denom = amp[idx].sum()
        boots[b] = win[idx].sum() / denom if denom > 0 else np.nan

    boots = boots[np.isfinite(boots)]
    lo, hi = np.quantile(boots, [0.025, 0.975])

    return {
        "fraction": point,
        "ci95_low": float(lo),
        "ci95_high": float(hi),
        "queries_with_amplification": int(n),
        "amplification_events": int(amp.sum()),
        "win_events": int(win.sum()),
    }


# ============================================================
# A1 — DIRECT PAIRED MECHANISM CONTRAST
# representation H3abs - nprobe H3abs
# ============================================================

print("\n" + "=" * 88)
print("A1 — DIRECT PAIRED MECHANISM CONTRAST")
print("=" * 88)

# Average all policies within query × mechanism first.
qm = (
    ep.groupby(["query_id", "mechanism"], as_index=False)
    ["H3_abs_slope"]
    .mean()
)

wide = qm.pivot(
    index="query_id",
    columns="mechanism",
    values="H3_abs_slope",
).dropna()

assert len(wide) == 3490

wide["delta_rep_minus_nprobe"] = (
    wide["representation"] - wide["nprobe"]
)

A1 = percentile_ci(
    wide["delta_rep_minus_nprobe"].to_numpy(),
    reps=BOOTSTRAP_REPS,
    seed=SEED + 2410,
)

print("representation mean:",
      f"{wide['representation'].mean():+.8f}")
print("nprobe mean:",
      f"{wide['nprobe'].mean():+.8f}")
print("paired delta (rep - nprobe):",
      f"{A1['mean']:+.8f}")
print(
    "95% query-bootstrap CI:",
    f"[{A1['ci95_low']:+.8f}, {A1['ci95_high']:+.8f}]"
)
print("n_queries:", A1["n_queries"])
print(
    "CI excludes zero:",
    bool(A1["ci95_low"] > 0 or A1["ci95_high"] < 0)
)

# ============================================================
# A2 — MEAN-ONLY vs SOFTMAX-ONLY
# ============================================================

print("\n" + "=" * 88)
print("A2 — FEEDBACK-FAMILY DECOMPOSITION")
print("=" * 88)

family_rows = []

for mech in ["representation", "nprobe"]:
    for method in ["mean", "softmax"]:

        sub = ep[
            (ep["mechanism"] == mech)
            & (ep["method"] == method)
        ]

        # Query is primary statistical unit.
        qvals = (
            sub.groupby("query_id")["H3_abs_slope"]
            .mean()
            .dropna()
        )

        assert len(qvals) == 3490

        res = percentile_ci(
            qvals.to_numpy(),
            reps=BOOTSTRAP_REPS,
            seed=(
                SEED
                + (100 if mech == "representation" else 200)
                + (1 if method == "mean" else 2)
            ),
        )

        family_rows.append({
            "mechanism": mech,
            "method": method,
            **res,
        })

family = pd.DataFrame(family_rows)

print(
    family[
        [
            "mechanism",
            "method",
            "mean",
            "ci95_low",
            "ci95_high",
            "n_queries",
        ]
    ].to_string(index=False)
)

print("\nDirectional audit:")
for _, r in family.iterrows():
    expected = (
        r["mean"] > 0
        if r["mechanism"] == "representation"
        else r["mean"] < 0
    )

    print(
        f"{r['mechanism']:15s} / {r['method']:7s} "
        f"H3abs={r['mean']:+.8f} | "
        f"expected-sign match={expected}"
    )


# ------------------------------------------------------------
# Optional stronger test:
# within each feedback family,
# directly compare representation - nprobe per query.
# ------------------------------------------------------------

family_delta_rows = []

for method in ["mean", "softmax"]:

    sub = (
        ep[ep["method"] == method]
        .groupby(["query_id", "mechanism"], as_index=False)
        ["H3_abs_slope"]
        .mean()
    )

    w = sub.pivot(
        index="query_id",
        columns="mechanism",
        values="H3_abs_slope",
    ).dropna()

    d = w["representation"] - w["nprobe"]

    res = percentile_ci(
        d.to_numpy(),
        reps=BOOTSTRAP_REPS,
        seed=SEED + (2421 if method == "mean" else 2422),
    )

    family_delta_rows.append({
        "method": method,
        **res,
    })

family_delta = pd.DataFrame(family_delta_rows)

print("\nPaired mechanism contrast within feedback family:")
print(family_delta.to_string(index=False))


# ============================================================
# A3 — SIGNED DIRECTION WITH QUERY-CLUSTER CI
# ============================================================

print("\n" + "=" * 88)
print("A3 — SIGNED DIRECTION CONDITIONAL ON AMPLIFICATION")
print("=" * 88)

signed_rows = []

for mech in ["representation", "nprobe"]:

    sub = ep[ep["mechanism"] == mech].copy()

    sub["is_amplification"] = (
        sub["H3_abs_slope"] > EPS_PRIMARY
    )

    # higher fidelity is beneficial when final_signed_gap > 0
    sub["higher_fidelity_win"] = (
        sub["is_amplification"]
        & (sub["final_signed_gap"] > 0)
    )

    sub["lower_fidelity_win"] = (
        sub["is_amplification"]
        & (sub["final_signed_gap"] < 0)
    )

    sub["tie"] = (
        sub["is_amplification"]
        & (sub["final_signed_gap"] == 0)
    )

    amp = sub[sub["is_amplification"]]

    n_amp = len(amp)
    n_high = int((amp["final_signed_gap"] > 0).sum())
    n_low = int((amp["final_signed_gap"] < 0).sum())
    n_tie = int((amp["final_signed_gap"] == 0).sum())

    ci = cluster_bootstrap_event_fraction(
        sub,
        event_mask_col="is_amplification",
        win_col="higher_fidelity_win",
        reps=BOOTSTRAP_REPS,
        seed=SEED + (2431 if mech == "representation" else 2432),
    )

    signed_rows.append({
        "mechanism": mech,
        "epsilon": EPS_PRIMARY,
        "amplification_events": n_amp,
        "higher_fidelity_wins": n_high,
        "lower_fidelity_wins": n_low,
        "ties": n_tie,
        "higher_fidelity_win_fraction": (
            n_high / n_amp if n_amp else np.nan
        ),
        "lower_fidelity_win_fraction": (
            n_low / n_amp if n_amp else np.nan
        ),
        "tie_fraction": (
            n_tie / n_amp if n_amp else np.nan
        ),
        "cluster_ci95_low": ci["ci95_low"],
        "cluster_ci95_high": ci["ci95_high"],
        "queries_with_amplification": (
            ci["queries_with_amplification"]
        ),
    })

signed = pd.DataFrame(signed_rows)

print(signed.to_string(index=False))


# ============================================================
# A4 — Compact reviewer-facing table
# ============================================================

print("\n" + "=" * 88)
print("A4 — REVIEWER-FACING SUMMARY")
print("=" * 88)

review_rows = []

# global mechanism contrast
review_rows.append({
    "audit": "paired mechanism contrast",
    "setting": "all policies",
    "estimate": A1["mean"],
    "ci95_low": A1["ci95_low"],
    "ci95_high": A1["ci95_high"],
    "n_queries": A1["n_queries"],
})

# family-specific deltas
for _, r in family_delta.iterrows():
    review_rows.append({
        "audit": "paired mechanism contrast",
        "setting": r["method"],
        "estimate": r["mean"],
        "ci95_low": r["ci95_low"],
        "ci95_high": r["ci95_high"],
        "n_queries": r["n_queries"],
    })

# signed fractions
for _, r in signed.iterrows():
    review_rows.append({
        "audit": "HF win | amplification",
        "setting": r["mechanism"],
        "estimate": r["higher_fidelity_win_fraction"],
        "ci95_low": r["cluster_ci95_low"],
        "ci95_high": r["cluster_ci95_high"],
        "n_queries": r["queries_with_amplification"],
    })

review = pd.DataFrame(review_rows)

print(review.to_string(index=False))


# ============================================================
# Persist outputs
# ============================================================

AUDIT_DIR = OUT / "post_primary_reviewer_audit"
AUDIT_DIR.mkdir(exist_ok=True)

wide.reset_index().to_csv(
    AUDIT_DIR / "v024_paired_mechanism_query_values.csv",
    index=False,
)

family.to_csv(
    AUDIT_DIR / "v024_feedback_family_h3abs.csv",
    index=False,
)

family_delta.to_csv(
    AUDIT_DIR / "v024_feedback_family_mechanism_delta.csv",
    index=False,
)

signed.to_csv(
    AUDIT_DIR / "v024_signed_direction_cluster_ci.csv",
    index=False,
)

review.to_csv(
    AUDIT_DIR / "v024_reviewer_audit_summary.csv",
    index=False,
)

report = {
    "status": "POST_PRIMARY_REVIEWER_AUDIT",
    "primary_experiment_modified": False,
    "retrieval_rerun": False,
    "epsilon": EPS_PRIMARY,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "paired_mechanism_contrast": A1,
    "feedback_family": family.to_dict(orient="records"),
    "feedback_family_mechanism_delta":
        family_delta.to_dict(orient="records"),
    "signed_direction": signed.to_dict(orient="records"),
}

with open(
    AUDIT_DIR / "v024_reviewer_audit_report.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(report, f, indent=2)

print("\nSaved to:")
print(AUDIT_DIR)

print("\nFILES:")
for p in sorted(AUDIT_DIR.iterdir()):
    print(" -", p.name)

print("\nPOST-PRIMARY REVIEWER AUDIT — COMPLETE")